# EA1 – Carga y creación de dataset desde Hugging Face

**Analítica de Datos – IU Digital de Antioquia** · PREICA2602B020125 · AE_Grupo 12

En este notebook coordinamos dos clases de Python:

1. `CargadorHuggingFace` (`cargador_hf.py`): se conecta a Hugging Face y devuelve el dataset como DataFrame de pandas.
2. `ResumenDataFrame` (`resumen_dataframe.py`): recibe un DataFrame y guarda un resumen en un archivo `.txt`.

**Flujo:** clase de carga → DataFrame → transformaciones → clase de resumen → archivo `.txt`

**Dataset elegido:** [`aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026`](https://huggingface.co/datasets/aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026).
Compara el precio por metro cuadrado de casas y apartamentos en ciudades de Colombia (octubre de 2026).

## 1. Importar las clases

Las dos clases están en archivos `.py` dentro de la misma carpeta `EA1`, así que se importan directamente.

In [1]:
import pandas as pd

from cargador_hf import CargadorHuggingFace
from resumen_dataframe import ResumenDataFrame

/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Obtener el DataFrame desde Hugging Face

Creamos el objeto con el identificador del dataset y el split `train`.
Con `max_filas` dejamos un límite para no traer más datos de los necesarios (recomendación de la clase para no romper el repositorio).

In [2]:
DATASET = "aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026"

cargador = CargadorHuggingFace(DATASET, split="train", max_filas=1000)
df_original = cargador.cargar()

print(cargador.informacion())
df_original.head()

Generating train split:   0%|          | 0/38 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 38/38 [00:00<00:00, 8912.57 examples/s]

{'dataset': 'aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026', 'split': 'train', 'configuracion': None, 'filas': 38, 'columnas': 6}


,city,n_si,n_no,casa_cop_m2,apto_cop_m2,diferencia_pct
0,Santa Marta,1084,3779,3.490169e+06,8.720930e+06,-59.979399
1,Cartagena,1327,5774,4.296875e+06,7.857143e+06,-45.312500
2,Palmira,357,112,2.453416e+06,4.194668e+06,-41.511080
3,Bogotá,17756,31265,4.686645e+06,7.261905e+06,-35.462599
4,Puerto Colombia,139,144,3.938115e+06,5.867102e+06,-32.878019


In [3]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   city            38 non-null     object 
 1   n_si            38 non-null     int64  
 2   n_no            38 non-null     int64  
 3   casa_cop_m2     38 non-null     float64
 4   apto_cop_m2     38 non-null     float64
 5   diferencia_pct  38 non-null     float64
dtypes: float64(3), int64(2), object(1)
memory usage: 1.9+ KB


## 3. Transformaciones del DataFrame

Cada transformación devuelve un DataFrame nuevo, así el original queda intacto.

### Transformación 1: renombrar columnas

Los nombres originales mezclan inglés y abreviaturas. Los cambiamos por nombres en español más claros.

In [4]:
df_t1 = df_original.rename(columns={
    "city": "ciudad",
    "casa_cop_m2": "precio_casa_m2",
    "apto_cop_m2": "precio_apto_m2",
    "diferencia_pct": "diferencia_casa_vs_apto_pct",
})
df_t1.columns.tolist()

['ciudad',
 'n_si',
 'n_no',
 'precio_casa_m2',
 'precio_apto_m2',
 'diferencia_casa_vs_apto_pct']

### Transformación 2: seleccionar columnas

Nos quedamos solo con las columnas que vamos a analizar. Las columnas `n_si` y `n_no` se quitan porque
la ficha del dataset no explica qué significan, y no tiene sentido analizar algo que no sabemos interpretar.

In [5]:
df_t2 = df_t1[["ciudad", "precio_casa_m2", "precio_apto_m2", "diferencia_casa_vs_apto_pct"]]
print(f"Columnas antes: {df_t1.shape[1]} | Columnas después: {df_t2.shape[1]}")
df_t2.head()

Columnas antes: 6 | Columnas después: 4


,ciudad,precio_casa_m2,precio_apto_m2,diferencia_casa_vs_apto_pct
0,Santa Marta,3.490169e+06,8.720930e+06,-59.979399
1,Cartagena,4.296875e+06,7.857143e+06,-45.312500
2,Palmira,2.453416e+06,4.194668e+06,-41.511080
3,Bogotá,4.686645e+06,7.261905e+06,-35.462599
4,Puerto Colombia,3.938115e+06,5.867102e+06,-32.878019


### Transformación 3: crear columnas nuevas

- Pasamos los precios a **millones de pesos** (redondeados a 2 decimales) para que sean más fáciles de leer.
- Creamos `tipo_mas_economico`, que indica si en esa ciudad sale más barato el m² de casa o de apartamento
  (si la diferencia es negativa, la casa es más barata).

In [6]:
df_t3 = df_t2.assign(
    precio_casa_m2_millones=(df_t2["precio_casa_m2"] / 1_000_000).round(2),
    precio_apto_m2_millones=(df_t2["precio_apto_m2"] / 1_000_000).round(2),
    diferencia_casa_vs_apto_pct=df_t2["diferencia_casa_vs_apto_pct"].round(2),
    tipo_mas_economico=df_t2["diferencia_casa_vs_apto_pct"].apply(
        lambda dif: "Sin dato" if pd.isna(dif) else ("Casa" if dif < 0 else "Apartamento")
    ),
)
df_t3.head()

,ciudad,precio_casa_m2,precio_apto_m2,diferencia_casa_vs_apto_pct,precio_casa_m2_millones,precio_apto_m2_millones,tipo_mas_economico
0,Santa Marta,3.490169e+06,8.720930e+06,-59.98,3.49,8.72,Casa
1,Cartagena,4.296875e+06,7.857143e+06,-45.31,4.30,7.86,Casa
2,Palmira,2.453416e+06,4.194668e+06,-41.51,2.45,4.19,Casa
3,Bogotá,4.686645e+06,7.261905e+06,-35.46,4.69,7.26,Casa
4,Puerto Colombia,3.938115e+06,5.867102e+06,-32.88,3.94,5.87,Casa


### Transformación 4: quitar columnas repetidas y ordenar

Como ya tenemos los precios en millones, eliminamos las columnas en pesos y ordenamos las ciudades
de mayor a menor diferencia a favor de la casa (las más negativas primero).

In [7]:
df_transformado = (
    df_t3.drop(columns=["precio_casa_m2", "precio_apto_m2"])
         .sort_values("diferencia_casa_vs_apto_pct")
         .reset_index(drop=True)
)
df_transformado.head(10)

,ciudad,diferencia_casa_vs_apto_pct,precio_casa_m2_millones,precio_apto_m2_millones,tipo_mas_economico
0,Santa Marta,-59.98,3.49,8.72,Casa
1,Cartagena,-45.31,4.30,7.86,Casa
2,Palmira,-41.51,2.45,4.19,Casa
3,Bogotá,-35.46,4.69,7.26,Casa
4,Puerto Colombia,-32.88,3.94,5.87,Casa
5,Manizales,-31.97,3.85,5.65,Casa
6,Sabaneta,-29.79,5.00,7.12,Casa
7,Montería,-29.41,2.62,3.72,Casa
8,Villavicencio,-27.41,3.05,4.20,Casa
9,Cali,-26.88,3.40,4.65,Casa


### Comparación antes y después

In [8]:
print(f"Original:     {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Transformado: {df_transformado.shape[0]} filas x {df_transformado.shape[1]} columnas")
print("Columnas finales:", df_transformado.columns.tolist())

Original:     38 filas x 6 columnas
Transformado: 38 filas x 5 columnas
Columnas finales: ['ciudad', 'diferencia_casa_vs_apto_pct', 'precio_casa_m2_millones', 'precio_apto_m2_millones', 'tipo_mas_economico']


## 4. Generar el resumen en `.txt`

Le pasamos el DataFrame **transformado** a la segunda clase y guardamos el archivo en la carpeta `resultados`.

In [9]:
resumen = ResumenDataFrame(
    df_transformado,
    titulo="Resumen: precio m2 casa vs apartamento en ciudades de Colombia (2026)",
)
ruta_resumen = resumen.guardar("resultados/resumen_casa_vs_apartamento.txt")

print(f"Resumen guardado en: {ruta_resumen}\n")
print(ruta_resumen.read_text(encoding="utf-8"))

Resumen guardado en: resultados/resumen_casa_vs_apartamento.txt

Resumen: precio m2 casa vs apartamento en ciudades de Colombia (2026)
Generado: 2026-10-21 14:35

DIMENSIONES
Filas: 38
Columnas: 5
Filas duplicadas: 0

COLUMNAS Y TIPOS DE DATO
- ciudad: object
- diferencia_casa_vs_apto_pct: float64
- precio_casa_m2_millones: float64
- precio_apto_m2_millones: float64
- tipo_mas_economico: object

VALORES NULOS POR COLUMNA
- ciudad: 0
- diferencia_casa_vs_apto_pct: 0
- precio_casa_m2_millones: 0
- precio_apto_m2_millones: 0
- tipo_mas_economico: 0
Total de valores nulos: 0

ESTADÍSTICAS DESCRIPTIVAS (columnas numéricas)
       diferencia_casa_vs_apto_pct  precio_casa_m2_millones  precio_apto_m2_millones
count                        38.00                    38.00                    38.00
mean                        -17.43                     4.44                     5.49
std                          16.57                     1.28                     1.58
min                         -59.98

## 5. Resultado y explicación

**Transformaciones aplicadas:**

1. **Renombrar** las columnas a nombres claros en español.
2. **Seleccionar** solo las columnas útiles (se quitaron `n_si` y `n_no` porque no están documentadas).
3. **Crear columnas nuevas:** precios en millones de pesos y la columna `tipo_mas_economico`.
4. **Eliminar columnas repetidas y ordenar** las ciudades por la diferencia de precio.

El archivo `resultados/resumen_casa_vs_apartamento.txt` corresponde al DataFrame ya transformado.

In [10]:
conteo = df_transformado["tipo_mas_economico"].value_counts()
print("Ciudades donde el m2 es más barato según el tipo de vivienda:")
print(conteo.to_string())

mayor = df_transformado.iloc[0]
print(f"\nMayor diferencia a favor de la casa: {mayor['ciudad']} "
      f"({mayor['diferencia_casa_vs_apto_pct']}%)")

Ciudades donde el m2 es más barato según el tipo de vivienda:
tipo_mas_economico
Casa           33
Apartamento     5

Mayor diferencia a favor de la casa: Santa Marta (-59.98%)
